In [1]:
import glob
import os
from dataclasses import dataclass
from typing import Optional, Sequence, Dict, Tuple

import numpy as np
import pandas as pd
import xarray as xr
from scipy.spatial import cKDTree


In [2]:
@dataclass
class VariableSpec:
    """Specification for one analysis variable."""
    name: str
    level_index: Optional[int] = None   # None for 2D variables like PS
    dim_name: Optional[str] = None      # e.g. "lev", "levtot"


class AtmosLandCrossCorrelationAnalyzer:
    """
    Analyze ensemble-based atmosphere-land cross correlations using:
      - atmosphere on native EAM ne30 ncol
      - land aggregated from ELM column -> gridcell

    Main workflow:
      1) discover paired eam.i / elm.r files
      2) aggregate land columns to land gridcells
      3) build nearest-neighbor mapping land_gridcell -> atm_ncol
      4) compute ensemble cross correlations
    """

    def __init__(
        self,
        base_dir: str,
        time_tag: str,
        ensemble_ids: Optional[Sequence[str]] = None,
    ):
        self.base_dir = base_dir
        self.time_tag = time_tag
        self.ensemble_ids = ensemble_ids

        self.eam_files: Dict[str, str] = {}
        self.elm_files: Dict[str, str] = {}

        self.land_gridcell_ids: Optional[np.ndarray] = None
        self.land_lat: Optional[np.ndarray] = None
        self.land_lon: Optional[np.ndarray] = None

        self.atm_lat: Optional[np.ndarray] = None
        self.atm_lon: Optional[np.ndarray] = None

        self.land_to_atm_index: Optional[np.ndarray] = None
        self.land_to_atm_distance_km: Optional[np.ndarray] = None

        self._discover_files()

    # ------------------------------------------------------------------
    # File discovery
    # ------------------------------------------------------------------
    def _discover_files(self):
        eam_pattern = os.path.join(self.base_dir, f"*.eam.i.{self.time_tag}.nc")
        elm_pattern = os.path.join(self.base_dir, f"*.elm.r.{self.time_tag}.nc")

        eam_all = sorted(glob.glob(eam_pattern))
        elm_all = sorted(glob.glob(elm_pattern))

        def extract_ens_id(path: str) -> str:
            fname = os.path.basename(path)
            parts = fname.split(".")
            for p in parts:
                if p.startswith("EN"):
                    return p
            raise ValueError(f"Could not parse ensemble ID from {fname}")

        eam_map = {extract_ens_id(f): f for f in eam_all}
        elm_map = {extract_ens_id(f): f for f in elm_all}

        common_ids = sorted(set(eam_map) & set(elm_map))
        if self.ensemble_ids is not None:
            common_ids = [eid for eid in self.ensemble_ids if eid in common_ids]

        if not common_ids:
            raise ValueError("No matched eam.i / elm.r ensemble pairs found.")

        self.eam_files = {eid: eam_map[eid] for eid in common_ids}
        self.elm_files = {eid: elm_map[eid] for eid in common_ids}

        print(f"Found {len(common_ids)} matched ensemble pairs.")

    # ------------------------------------------------------------------
    # Coordinate helpers
    # ------------------------------------------------------------------
    @staticmethod
    def _normalize_lon(lon: np.ndarray) -> np.ndarray:
        """Convert longitude to [-180, 180)."""
        lon = np.asarray(lon, dtype=float)
        return ((lon + 180.0) % 360.0) - 180.0

    @staticmethod
    def _latlon_to_xyz(lat_deg: np.ndarray, lon_deg: np.ndarray) -> np.ndarray:
        """Convert lat/lon to unit-sphere Cartesian coordinates."""
        lat = np.deg2rad(lat_deg)
        lon = np.deg2rad(lon_deg)
        x = np.cos(lat) * np.cos(lon)
        y = np.cos(lat) * np.sin(lon)
        z = np.sin(lat)
        return np.column_stack((x, y, z))

    @staticmethod
    def _great_circle_distance_km(
        lat1: np.ndarray, lon1: np.ndarray, lat2: np.ndarray, lon2: np.ndarray
    ) -> np.ndarray:
        """Vectorized great-circle distance in km."""
        r_earth = 6371.0
        lat1 = np.deg2rad(lat1)
        lon1 = np.deg2rad(lon1)
        lat2 = np.deg2rad(lat2)
        lon2 = np.deg2rad(lon2)

        dlat = lat2 - lat1
        dlon = lon2 - lon1

        a = np.sin(dlat / 2.0) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2.0) ** 2
        c = 2.0 * np.arcsin(np.sqrt(a))
        return r_earth * c

    @staticmethod
    def _infer_atm_latlon_names(ds: xr.Dataset) -> Tuple[str, str]:
        candidates = [
            ("lat", "lon"),
            ("lats", "lons"),
            ("grid_center_lat", "grid_center_lon"),
        ]
        for lat_name, lon_name in candidates:
            if lat_name in ds.variables and lon_name in ds.variables:
                return lat_name, lon_name
        raise ValueError("Could not infer atmosphere lat/lon variable names.")

    # ------------------------------------------------------------------
    # Low-level extraction helpers
    # ------------------------------------------------------------------
    @staticmethod
    def _extract_1d_column_values(ds: xr.Dataset, var_spec: VariableSpec) -> np.ndarray:
        """
        Extract a land variable as a 1D numpy array over 'column',
        avoiding xarray coordinate slicing issues.
        """
        da = ds[var_spec.name]
        dims = da.dims
        arr = np.asarray(da.values)

        if "time" in dims:
            tidx = dims.index("time")
            if arr.shape[tidx] < 1:
                raise ValueError(f"{var_spec.name}: time dimension has length 0.")
            arr = np.take(arr, indices=0, axis=tidx)
            dims = tuple(d for d in dims if d != "time")

        if var_spec.level_index is not None:
            if var_spec.dim_name is None:
                raise ValueError(f"{var_spec.name}: dim_name is required when level_index is given.")
            if var_spec.dim_name not in dims:
                raise ValueError(
                    f"{var_spec.name}: dim '{var_spec.dim_name}' not found after time slicing. "
                    f"Remaining dims: {dims}"
                )
            lidx = dims.index(var_spec.dim_name)
            if arr.shape[lidx] <= var_spec.level_index:
                raise IndexError(
                    f"{var_spec.name}: level_index={var_spec.level_index} out of bounds "
                    f"for dim '{var_spec.dim_name}' with size {arr.shape[lidx]}."
                )
            arr = np.take(arr, indices=var_spec.level_index, axis=lidx)
            dims = tuple(d for d in dims if d != var_spec.dim_name)

        if dims != ("column",):
            raise ValueError(
                f"{var_spec.name} did not reduce to 1D column field. Remaining dims: {dims}"
            )

        return np.asarray(arr)

    @staticmethod
    def _extract_1d_ncol_values(ds: xr.Dataset, var_spec: VariableSpec, source_name: str = "") -> np.ndarray:
        """
        Extract an atmosphere variable as a 1D numpy array over 'ncol',
        avoiding xarray coordinate slicing issues.
        """
        da = ds[var_spec.name]
        dims = da.dims
        arr = np.asarray(da.values)
    
        if "time" in dims:
            tidx = dims.index("time")
            if arr.shape[tidx] < 1:
                raise ValueError(
                    f"{source_name}: variable {var_spec.name} has time dimension of length 0. "
                    f"dims={dims}, shape={arr.shape}"
                )
            arr = np.take(arr, indices=0, axis=tidx)
            dims = tuple(d for d in dims if d != "time")
    
        if var_spec.level_index is not None:
            if var_spec.dim_name is None:
                raise ValueError(f"{source_name}: {var_spec.name}: dim_name is required when level_index is given.")
            if var_spec.dim_name not in dims:
                raise ValueError(
                    f"{source_name}: {var_spec.name}: dim '{var_spec.dim_name}' not found after time slicing. "
                    f"Remaining dims: {dims}"
                )
            lidx = dims.index(var_spec.dim_name)
            if arr.shape[lidx] <= var_spec.level_index:
                raise IndexError(
                    f"{source_name}: {var_spec.name}: level_index={var_spec.level_index} out of bounds "
                    f"for dim '{var_spec.dim_name}' with size {arr.shape[lidx]}."
                )
            arr = np.take(arr, indices=var_spec.level_index, axis=lidx)
            dims = tuple(d for d in dims if d != var_spec.dim_name)
    
        if dims != ("ncol",):
            raise ValueError(
                f"{source_name}: {var_spec.name} did not reduce to 1D ncol field. "
                f"Remaining dims: {dims}, shape={arr.shape}"
            )
    
        return np.asarray(arr)

    # ------------------------------------------------------------------
    # Land aggregation
    # ------------------------------------------------------------------
    def _aggregate_elm_to_gridcell(
        self,
        ds_elm: xr.Dataset,
        var_spec: VariableSpec,
        active_only: bool = True,
    ) -> pd.DataFrame:
        """
        Aggregate one ELM variable from column -> gridcell.
        Returns a DataFrame with columns: gc, lat, lon, value
        """
        values = self._extract_1d_column_values(ds_elm, var_spec)

        active = (
            ds_elm["cols1d_active"].values == 1
            if active_only
            else np.ones(ds_elm.sizes["column"], dtype=bool)
        )
        gc = ds_elm["cols1d_gridcell_index"].values.astype(int)

        lat = np.asarray(ds_elm["cols1d_lat"].values)
        lon = self._normalize_lon(np.asarray(ds_elm["cols1d_lon"].values))

        df = pd.DataFrame(
            {
                "gc": gc[active],
                "lat": lat[active],
                "lon": lon[active],
                "value": values[active],
            }
        )

        out = (
            df.groupby("gc", as_index=False)
            .agg(
                {
                    "lat": "mean",
                    "lon": "mean",
                    "value": "mean",
                }
            )
            .sort_values("gc")
            .reset_index(drop=True)
        )
        return out

    def build_land_grid(self, land_var: VariableSpec, active_only: bool = True):
        """Build static land gridcell coordinates from the first ensemble file."""
        first_eid = next(iter(self.elm_files))
        with xr.open_dataset(self.elm_files[first_eid]) as ds_elm:
            land_df = self._aggregate_elm_to_gridcell(ds_elm, land_var, active_only=active_only)

        self.land_gridcell_ids = land_df["gc"].to_numpy()
        self.land_lat = land_df["lat"].to_numpy()
        self.land_lon = land_df["lon"].to_numpy()

        print(f"Built land grid with {len(self.land_gridcell_ids)} aggregated gridcells.")

    # ------------------------------------------------------------------
    # Atmosphere grid + nearest-neighbor map
    # ------------------------------------------------------------------
    def build_land_to_atm_mapping(self):
        """Build nearest-neighbor map from aggregated land gridcells to atmosphere ncol."""
        if self.land_lat is None or self.land_lon is None:
            raise RuntimeError("Call build_land_grid() first.")

        first_eid = next(iter(self.eam_files))
        with xr.open_dataset(self.eam_files[first_eid]) as ds_eam:
            lat_name, lon_name = self._infer_atm_latlon_names(ds_eam)
            self.atm_lat = np.asarray(ds_eam[lat_name].values)
            self.atm_lon = self._normalize_lon(np.asarray(ds_eam[lon_name].values))

        atm_xyz = self._latlon_to_xyz(self.atm_lat, self.atm_lon)
        land_xyz = self._latlon_to_xyz(self.land_lat, self.land_lon)

        tree = cKDTree(atm_xyz)
        _, idx = tree.query(land_xyz, k=1)

        self.land_to_atm_index = idx.astype(int)
        self.land_to_atm_distance_km = self._great_circle_distance_km(
            self.land_lat,
            self.land_lon,
            self.atm_lat[self.land_to_atm_index],
            self.atm_lon[self.land_to_atm_index],
        )

        print("Built land -> nearest atmosphere mapping.")
        print(f"Mean distance: {np.nanmean(self.land_to_atm_distance_km):.2f} km")
        print(f"Max distance : {np.nanmax(self.land_to_atm_distance_km):.2f} km")

    # ------------------------------------------------------------------
    # Ensemble extraction
    # ------------------------------------------------------------------
    def get_land_ensemble_matrix(
        self,
        land_var: VariableSpec,
        active_only: bool = True,
    ) -> xr.DataArray:
        """
        Return land values as DataArray with dims:
            ensemble, land_point
        """
        if self.land_gridcell_ids is None:
            raise RuntimeError("Call build_land_grid() first.")

        rows = []
        ensemble_ids = []

        for eid in self.elm_files:
            with xr.open_dataset(self.elm_files[eid]) as ds_elm:
                land_df = self._aggregate_elm_to_gridcell(ds_elm, land_var, active_only=active_only)

            land_df = land_df.set_index("gc").reindex(self.land_gridcell_ids)
            rows.append(land_df["value"].to_numpy())
            ensemble_ids.append(eid)

        arr = np.vstack(rows)
        return xr.DataArray(
            arr,
            dims=("ensemble", "land_point"),
            coords={
                "ensemble": ensemble_ids,
                "land_point": np.arange(arr.shape[1]),
                "gridcell": ("land_point", self.land_gridcell_ids),
                "lat": ("land_point", self.land_lat),
                "lon": ("land_point", self.land_lon),
            },
            name=land_var.name,
        )

    def get_atm_ensemble_matrix(
        self,
        atm_var: VariableSpec,
    ) -> xr.DataArray:
        """
        Return atmosphere values as DataArray with dims:
            ensemble, ncol
        """
        if self.atm_lat is None or self.atm_lon is None:
            raise RuntimeError("Call build_land_to_atm_mapping() first.")
    
        rows = []
        ensemble_ids = []
    
        for eid in self.eam_files:
            fpath = self.eam_files[eid]
            try:
                with xr.open_dataset(fpath, decode_times=False) as ds_eam:
                    vals = self._extract_1d_ncol_values(ds_eam, atm_var, source_name=f"{eid} | {fpath}")
            except Exception as e:
                raise RuntimeError(f"Failed reading atmosphere variable {atm_var.name} from {eid}: {fpath}\n{e}") from e
    
            rows.append(vals)
            ensemble_ids.append(eid)
    
        arr = np.vstack(rows)
        return xr.DataArray(
            arr,
            dims=("ensemble", "ncol"),
            coords={
                "ensemble": ensemble_ids,
                "ncol": np.arange(arr.shape[1]),
                "lat": ("ncol", self.atm_lat),
                "lon": ("ncol", self.atm_lon),
            },
            name=atm_var.name,
        )

    def filter_valid_ensembles(
        self,
        atm_var: VariableSpec,
        land_var: Optional[VariableSpec] = None,
        active_only: bool = True,
        verbose: bool = True,
    ):
        """
        Remove ensemble members whose atmosphere or land files are invalid
        for the requested variables.
        """
        valid_eam = {}
        valid_elm = {}
    
        for eid in list(self.eam_files.keys()):
            eam_ok = True
            elm_ok = True
    
            # Check atmosphere file
            try:
                with xr.open_dataset(self.eam_files[eid], decode_times=False) as ds_eam:
                    _ = self._extract_1d_ncol_values(
                        ds_eam, atm_var, source_name=f"{eid} | {self.eam_files[eid]}"
                    )
            except Exception as e:
                eam_ok = False
                if verbose:
                    print(f"[DROP ATM] {eid}: {e}")
    
            # Check land file if requested
            if land_var is not None:
                try:
                    with xr.open_dataset(self.elm_files[eid], decode_times=False) as ds_elm:
                        _ = self._aggregate_elm_to_gridcell(ds_elm, land_var, active_only=active_only)
                except Exception as e:
                    elm_ok = False
                    if verbose:
                        print(f"[DROP LND] {eid}: {e}")
    
            if eam_ok and elm_ok:
                valid_eam[eid] = self.eam_files[eid]
                valid_elm[eid] = self.elm_files[eid]
    
        self.eam_files = valid_eam
        self.elm_files = valid_elm
    
        print(f"Retained {len(self.eam_files)} valid ensemble pairs.")
        
    # ------------------------------------------------------------------
    # Correlation methods
    # ------------------------------------------------------------------
    @staticmethod
    def _corr_2d(x: np.ndarray, y: np.ndarray) -> np.ndarray:
        """
        Correlate x and y across ensemble axis (axis=0).
        x, y shape: (nens, npoint)
        Returns r(point), shape (npoint,)
        """
        x = x - np.nanmean(x, axis=0, keepdims=True)
        y = y - np.nanmean(y, axis=0, keepdims=True)

        num = np.nansum(x * y, axis=0)
        den = np.sqrt(np.nansum(x * x, axis=0) * np.nansum(y * y, axis=0))

        r = np.full(num.shape, np.nan, dtype=float)
        mask = den > 0
        r[mask] = num[mask] / den[mask]
        return r

    def compute_collocated_correlation(
        self,
        atm_var: VariableSpec,
        land_var: VariableSpec,
        active_only: bool = True,
    ) -> xr.Dataset:
        """
        Compute correlation across ensemble members between:
          - land variable at each aggregated land gridcell
          - nearest collocated atmosphere variable at matched atm ncol
        """
        if self.land_to_atm_index is None:
            raise RuntimeError("Call build_land_to_atm_mapping() first.")

        land_da = self.get_land_ensemble_matrix(land_var, active_only=active_only)
        atm_da = self.get_atm_ensemble_matrix(atm_var)

        atm_collocated = atm_da.isel(ncol=xr.DataArray(self.land_to_atm_index, dims="land_point"))
        atm_collocated = atm_collocated.assign_coords(
            land_point=("land_point", np.arange(len(self.land_to_atm_index)))
        )
        atm_collocated = atm_collocated.transpose("ensemble", "land_point")

        r = self._corr_2d(atm_collocated.values, land_da.values)

        ds_out = xr.Dataset(
            {
                "correlation": ("land_point", r),
                "atm_ncol": ("land_point", self.land_to_atm_index),
                "distance_km": ("land_point", self.land_to_atm_distance_km),
                "land_value_mean": ("land_point", np.nanmean(land_da.values, axis=0)),
                "atm_value_mean": ("land_point", np.nanmean(atm_collocated.values, axis=0)),
            },
            coords={
                "land_point": np.arange(len(r)),
                "gridcell": ("land_point", self.land_gridcell_ids),
                "lat": ("land_point", self.land_lat),
                "lon": ("land_point", self.land_lon),
            },
            attrs={
                "atm_var": atm_var.name,
                "land_var": land_var.name,
                "atm_level_index": atm_var.level_index if atm_var.level_index is not None else -1,
                "land_level_index": land_var.level_index if land_var.level_index is not None else -1,
            },
        )
        return ds_out

    def compute_single_landpoint_to_all_atm_correlation(
        self,
        atm_var: VariableSpec,
        land_var: VariableSpec,
        land_point_index: int,
        active_only: bool = True,
    ) -> xr.Dataset:
        """
        For one land point, correlate its ensemble perturbation with ALL atmosphere ncol points.
        Useful for localization-distance diagnostics.
        """
        land_da = self.get_land_ensemble_matrix(land_var, active_only=active_only)
        atm_da = self.get_atm_ensemble_matrix(atm_var)

        y = land_da.isel(land_point=land_point_index).values[:, None]
        x = atm_da.values
        r = self._corr_2d(x, np.repeat(y, x.shape[1], axis=1))

        dist = self._great_circle_distance_km(
            np.full_like(self.atm_lat, self.land_lat[land_point_index], dtype=float),
            np.full_like(self.atm_lon, self.land_lon[land_point_index], dtype=float),
            self.atm_lat,
            self.atm_lon,
        )

        return xr.Dataset(
            {
                "correlation": ("ncol", r),
                "distance_km": ("ncol", dist),
            },
            coords={
                "ncol": np.arange(len(r)),
                "lat": ("ncol", self.atm_lat),
                "lon": ("ncol", self.atm_lon),
            },
            attrs={
                "land_point_index": int(land_point_index),
                "land_gridcell": int(self.land_gridcell_ids[land_point_index]),
                "land_lat": float(self.land_lat[land_point_index]),
                "land_lon": float(self.land_lon[land_point_index]),
                "atm_var": atm_var.name,
                "land_var": land_var.name,
            },
        )

In [3]:
if __name__ == "__main__":
    data_dir = "/compyfs/zhan391/v3_dart_cda_scratch"
    case_name = "DARTEN40_F20TR_ne30pg2_r05_IcoswISC30E3r5_compy"
    sub_dir = "archive/post/init_rgd"
    time_tag = "2011-12-18-00000"

    base_dir = f"{data_dir}/{case_name}/{sub_dir}/{time_tag}"
    out_dir = f"{data_dir}/{case_name}/archive/post/cross-corr/{time_tag}"
    os.makedirs(out_dir, exist_ok=True)

    anl = AtmosLandCrossCorrelationAnalyzer(base_dir=base_dir, time_tag=time_tag)

    # Use one representative pair to filter bad ensembles once
    atm_check = VariableSpec(name="T", level_index=71, dim_name="lev")
    land_check = VariableSpec(name="T_SOISNO", level_index=5, dim_name="levtot")

    anl.filter_valid_ensembles(atm_var=atm_check, land_var=land_check, active_only=True)

    # Build static mapping once
    anl.build_land_grid(land_var=land_check, active_only=True)
    anl.build_land_to_atm_mapping()

    n_atm_lev = 80
    n_land_lev = 20
    land_name = "T_SOISNO"
    for atm_name in ["T", "Q"]:
        for atm_lev in [79]: #range(80):
            atm_var = VariableSpec(name=atm_name, level_index=atm_lev, dim_name="lev")
            for land_lev in range(20):
                land_var = VariableSpec(name=land_name, level_index=land_lev, dim_name="levtot")

                print(f"Running atm lev={atm_lev}, land levtot={land_lev}")
    
                ds_corr = anl.compute_collocated_correlation(
                    atm_var=atm_var,
                    land_var=land_var,
                    active_only=True,
                )
    
                out_file = (
                    f"{out_dir}/corr_{atm_name}_lev{atm_lev:02d}_vs_TSOISNO_levtot{land_lev:02d}_{time_tag}.nc"
                )
            
                if os.path.exists(out_file):
                    os.remove(out_file)
    
                ds_corr.to_netcdf(out_file, mode="w")
                print(f"Saved: {out_file}")

Found 40 matched ensemble pairs.
[DROP ATM] EN10: EN10 | /compyfs/zhan391/v3_dart_cda_scratch/DARTEN40_F20TR_ne30pg2_r05_IcoswISC30E3r5_compy/archive/post/init_rgd/2011-12-18-00000/DARTEN40_F20TR_ne30pg2_r05_IcoswISC30E3r5_compy.EN10.eam.i.2011-12-18-00000.nc: variable T has time dimension of length 0. dims=('time', 'lev', 'ncol'), shape=(0, 80, 48602)
[DROP ATM] EN18: EN18 | /compyfs/zhan391/v3_dart_cda_scratch/DARTEN40_F20TR_ne30pg2_r05_IcoswISC30E3r5_compy/archive/post/init_rgd/2011-12-18-00000/DARTEN40_F20TR_ne30pg2_r05_IcoswISC30E3r5_compy.EN18.eam.i.2011-12-18-00000.nc: variable T has time dimension of length 0. dims=('time', 'lev', 'ncol'), shape=(0, 80, 48602)
[DROP ATM] EN32: EN32 | /compyfs/zhan391/v3_dart_cda_scratch/DARTEN40_F20TR_ne30pg2_r05_IcoswISC30E3r5_compy/archive/post/init_rgd/2011-12-18-00000/DARTEN40_F20TR_ne30pg2_r05_IcoswISC30E3r5_compy.EN32.eam.i.2011-12-18-00000.nc: variable T has time dimension of length 0. dims=('time', 'lev', 'ncol'), shape=(0, 80, 48602)
[

/tmp/ipykernel_457910/4174543282.py:447: RuntimeWarning: Mean of empty slice
  y = y - np.nanmean(y, axis=0, keepdims=True)
/tmp/ipykernel_457910/4174543282.py:487: RuntimeWarning: Mean of empty slice
  "land_value_mean": ("land_point", np.nanmean(land_da.values, axis=0)),


Saved: /compyfs/zhan391/v3_dart_cda_scratch/DARTEN40_F20TR_ne30pg2_r05_IcoswISC30E3r5_compy/archive/post/cross-corr/2011-12-18-00000/corr_T_lev79_vs_TSOISNO_levtot00_2011-12-18-00000.nc
Running atm lev=79, land levtot=1


/tmp/ipykernel_457910/4174543282.py:447: RuntimeWarning: Mean of empty slice
  y = y - np.nanmean(y, axis=0, keepdims=True)
/tmp/ipykernel_457910/4174543282.py:487: RuntimeWarning: Mean of empty slice
  "land_value_mean": ("land_point", np.nanmean(land_da.values, axis=0)),


Saved: /compyfs/zhan391/v3_dart_cda_scratch/DARTEN40_F20TR_ne30pg2_r05_IcoswISC30E3r5_compy/archive/post/cross-corr/2011-12-18-00000/corr_T_lev79_vs_TSOISNO_levtot01_2011-12-18-00000.nc
Running atm lev=79, land levtot=2


...

Saved: /compyfs/zhan391/v3_dart_cda_scratch/DARTEN40_F20TR_ne30pg2_r05_IcoswISC30E3r5_compy/archive/post/cross-corr/2011-12-18-00000/corr_Q_lev79_vs_TSOISNO_levtot02_2011-12-18-00000.nc
Running atm lev=79, land levtot=3


/tmp/ipykernel_457910/4174543282.py:447: RuntimeWarning: Mean of empty slice
  y = y - np.nanmean(y, axis=0, keepdims=True)
/tmp/ipykernel_457910/4174543282.py:487: RuntimeWarning: Mean of empty slice
  "land_value_mean": ("land_point", np.nanmean(land_da.values, axis=0)),


Saved: /compyfs/zhan391/v3_dart_cda_scratch/DARTEN40_F20TR_ne30pg2_r05_IcoswISC30E3r5_compy/archive/post/cross-corr/2011-12-18-00000/corr_Q_lev79_vs_TSOISNO_levtot03_2011-12-18-00000.nc
Running atm lev=79, land levtot=4


/tmp/ipykernel_457910/4174543282.py:447: RuntimeWarning: Mean of empty slice
  y = y - np.nanmean(y, axis=0, keepdims=True)
/tmp/ipykernel_457910/4174543282.py:487: RuntimeWarning: Mean of empty slice
  "land_value_mean": ("land_point", np.nanmean(land_da.values, axis=0)),


Saved: /compyfs/zhan391/v3_dart_cda_scratch/DARTEN40_F20TR_ne30pg2_r05_IcoswISC30E3r5_compy/archive/post/cross-corr/2011-12-18-00000/corr_Q_lev79_vs_TSOISNO_levtot04_2011-12-18-00000.nc
Running atm lev=79, land levtot=5
Saved: /compyfs/zhan391/v3_dart_cda_scratch/DARTEN40_F20TR_ne30pg2_r05_IcoswISC30E3r5_compy/archive/post/cross-corr/2011-12-18-00000/corr_Q_lev79_vs_TSOISNO_levtot05_2011-12-18-00000.nc
Running atm lev=79, land levtot=6
Saved: /compyfs/zhan391/v3_dart_cda_scratch/DARTEN40_F20TR_ne30pg2_r05_IcoswISC30E3r5_compy/archive/post/cross-corr/2011-12-18-00000/corr_Q_lev79_vs_TSOISNO_levtot06_2011-12-18-00000.nc
Running atm lev=79, land levtot=7
Saved: /compyfs/zhan391/v3_dart_cda_scratch/DARTEN40_F20TR_ne30pg2_r05_IcoswISC30E3r5_compy/archive/post/cross-corr/2011-12-18-00000/corr_Q_lev79_vs_TSOISNO_levtot07_2011-12-18-00000.nc
Running atm lev=79, land levtot=8
Saved: /compyfs/zhan391/v3_dart_cda_scratch/DARTEN40_F20TR_ne30pg2_r05_IcoswISC30E3r5_compy/archive/post/cross-corr/2011